# Notebook 31 -- Luyben Data Generation

Generate the evaluation dataset:
- 12 fault scenarios × 30 replicates × 2-hour windows = 360 windows
- 2 control modes (closed-loop + open-loop ablation)
- Save to `data/luyben_observations.npz`


In [ ]:
import sys; sys.path.insert(0, '../src')
import jax
import jax.numpy as jnp
import numpy as np
from pathlib import Path

from cstr_sbi.luyben.scenarios import list_closed_loop_configs
from cstr_sbi.luyben.simulator import generate_replicates, warm_start_ic
from cstr_sbi.luyben.physics import NOMINAL_INLET, NOMINAL_CTRL_ALL

N_REPLICATES = 30
T_WINDOW = 120.0  # minutes
SEED_BASE = 42

print(f'Generating {len(list_closed_loop_configs())} scenarios × {N_REPLICATES} replicates')
print(f'Window length: {T_WINDOW} min | Output: 1 min resolution -> 120 timesteps')


In [ ]:
all_obs = []
all_theta = []
all_scenario_ids = []
all_t = None

for sc in list_closed_loop_configs():
    print(f'  Scenario {sc.id}: {sc.name} ...', end=' ', flush=True)
    theta = sc.theta()
    y0 = warm_start_ic(theta, NOMINAL_INLET, NOMINAL_CTRL_ALL)
    master_key = jax.random.PRNGKey(SEED_BASE + sc.id)
    t_out, obs = generate_replicates(
        theta, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0,
        n_replicates=N_REPLICATES,
        master_key=master_key,
        t_window=T_WINDOW,
    )
    all_obs.append(np.asarray(obs))         # (30, 120, 8)
    all_theta.append(np.tile(np.asarray(theta), (N_REPLICATES, 1)))  # (30, 8)
    all_scenario_ids.extend([sc.id] * N_REPLICATES)
    if all_t is None:
        all_t = np.asarray(t_out)
    print('done')

obs_arr    = np.concatenate(all_obs, axis=0)    # (360, 120, 8)
theta_arr  = np.concatenate(all_theta, axis=0)  # (360, 8)
sc_id_arr  = np.array(all_scenario_ids)         # (360,)

print(f'\nobs_arr shape:   {obs_arr.shape}')
print(f'theta_arr shape: {theta_arr.shape}')


In [ ]:
out_path = Path('../data/luyben_observations.npz')
out_path.parent.mkdir(exist_ok=True)
np.savez_compressed(
    out_path,
    x=obs_arr,
    theta=theta_arr,
    scenario_id=sc_id_arr,
    t=all_t,
)
print(f'Saved to {out_path}')
